Carga inicial del conjunto de datos utilizando el Parquet.

In [ ]:
!pip install pyarrow -q

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path

PARQUET_PATH = Path("/content/drive/MyDrive/data/processed/physionet_sepsis_raw_combined.parquet")

df = pd.read_parquet(PARQUET_PATH)

print("DataFrame cargado:", df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DataFrame cargado: (1552210, 44)


,PatientID,Hospital,TimeStep,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,p000001,A,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,p000001,A,2,97.0,95.0,NaN,98.0,75.33,NaN,19.0,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,p000001,A,3,89.0,99.0,NaN,122.0,86.00,NaN,22.0,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,p000001,A,4,90.0,95.0,NaN,NaN,NaN,NaN,30.0,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,p000001,A,5,103.0,88.5,NaN,122.0,91.33,NaN,24.5,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0


Separamos de forma clara las columnas auxiliares y objetivo de las columnas con las que se entrenará el modelo. `PatientID` no se usa como variable predictora, sino para agrupar registros por paciente. `SepsisLabel` es la variable objetivo. `Hospital` y `TimeStep` son variables auxiliares que ayudan al análisis y trazabilidad.

In [ ]:
target_col = "SepsisLabel"

aux_cols = ["PatientID", "Hospital", "TimeStep"]

feature_cols = [
    col for col in df.columns
    if col not in aux_cols + [target_col]
]

# 1. Particionado de los datos

No podemos dividir nuestro conjunto de datos en entrenamiento y test simplemente por filas, pues podría ocurrir entonces que registros de un mismo paciente se encuentren tanto en entrenamiento como en test. Esto sería fuga de información. Por ello, el particionado se hará a nivel de paciente y no a nivel de fila (o registro temporal).

In [ ]:
from sklearn.model_selection import train_test_split

# Creamos un DataFrame que resume la información por paciente
df_patient = df.groupby("PatientID").agg(
    TieneSepsis=("SepsisLabel", "max"),
    Hospital=("Hospital", "first"),
    NumRegistros=("SepsisLabel", "size")
).reset_index()

# Dividimos entre pacientes de entrenamiento y pacientes de prueba
# Fijamos random_state para asegurar la reproducibilidad del particionado
train_patients, test_patients = train_test_split(
    df_patient["PatientID"],
    test_size=0.2,
    random_state=42,
    stratify=df_patient["TieneSepsis"]
)

# Creamos una copia
df_train = df[df["PatientID"].isin(train_patients)].copy()
df_test = df[df["PatientID"].isin(test_patients)].copy()

In [ ]:
print("Pacientes train:", df_train["PatientID"].nunique())
print("Pacientes test:", df_test["PatientID"].nunique())

# Comprobamos que el desbalance de la clase de la variable objetivo es similar
# en ambos conjuntos, confirmando que la partición se ha realizado de forma
# estratificada
print("Sepsis pacientes train:", df_train.groupby("PatientID")["SepsisLabel"].max().mean() * 100)
print("Sepsis pacientes test:", df_test.groupby("PatientID")["SepsisLabel"].max().mean() * 100)

# Calculamos la proporción de pacientes en cada hospital
hospital_train = df_train.groupby("PatientID")["Hospital"].first().value_counts(normalize=True) * 100
hospital_test = df_test.groupby("PatientID")["Hospital"].first().value_counts(normalize=True) * 100

print("\n--- Distribución por Hospital en TRAIN ---")
print(f"Hospital A: {hospital_train.get('A', 0):.2f}%")
print(f"Hospital B: {hospital_train.get('B', 0):.2f}%")

print("\n--- Distribución por Hospital en TEST ---")
print(f"Hospital A: {hospital_test.get('A', 0):.2f}%")
print(f"Hospital B: {hospital_test.get('B', 0):.2f}%")

Pacientes train: 32268
Pacientes test: 8068
Sepsis pacientes train: 7.270360728895501
Sepsis pacientes test: 7.263262270699059

--- Distribución por Hospital en TRAIN ---
Hospital A: 50.43%
Hospital B: 49.57%

--- Distribución por Hospital en TEST ---
Hospital A: 50.36%
Hospital B: 49.64%


# 2. Análisis de valores ausentes en el conjunto de entrenamiento

Las decisiones de imputación deben tomarse usando solo el conjunto de entrenamiento. Por ello, hacemos un pequeño análisis aquí de los valores ausentes encontrados en el conjunto de entrenamiento.

In [ ]:
missing_train = df_train[feature_cols].isna().mean().sort_values(ascending=False) * 100

missing_summary = pd.DataFrame({
    "Variable": missing_train.index,
    "PorcentajeAusentesTrain": missing_train.values
})

missing_summary.head(20)

,Variable,PorcentajeAusentesTrain
0,Bilirubin_direct,99.804932
1,Fibrinogen,99.326856
2,TroponinI,99.046587
3,Bilirubin_total,98.502816
4,Alkalinephos,98.387146
5,AST,98.370702
6,Lactate,97.315806
7,PTT,97.023850
8,SaO2,96.516271
9,EtCO2,96.273243


In [ ]:
threshold_missing = 95

cols_to_drop_missing = missing_summary.loc[
    missing_summary["PorcentajeAusentesTrain"] > threshold_missing,
    "Variable"
].tolist()

cols_to_drop_missing

['Bilirubin_direct',
 'Fibrinogen',
 'TroponinI',
 'Bilirubin_total',
 'Alkalinephos',
 'AST',
 'Lactate',
 'PTT',
 'SaO2',
 'EtCO2',
 'Phosphate',
 'HCO3',
 'Chloride']

En vista de los resultados, se aplicarán diferentes estrategias para tratar con variables con valores ausentes, pues no todas ellas presentan la misma relevancia clínica.

### Variables auxiliares y objetivo
Se conservan las variables `PatientID`, `Hospital` y `TimeStep`, aunque no se usan como variables predictoras del modelo. Se conserva también `SepsisLabel`, que corresponde a la variable objetivo del problema.

### Variables de laboratorio
Las variables de laboratorio presentan porcentajes elevados de valores ausentes, por lo que no se aplica una única estrategia homogénea para todas ellas. En su lugar, se decide conservar aquellas variables con mayor relevancia clínica potencial para la detección de sepsis y descartar aquellas con ausencia extrema, menor utilidad práctica para el modelo base o elevada redundancia con otras variables.


*   `TroponinI` (troponina), se descarta por su porcentaje elevado de valores ausentes (>99%) y es un biomarcador miocárdico ([Guzmán y Quiroga, 2010](http://dx.doi.org/10.4067/S0034-98872010000300020)).


*   `Bilirubin_direct` se descarta por su ausencia extrema pero también por su redundancia con `Bilirubin_total`, conservando esta última por tener un menor número de valores ausentes y por ser más habitual para evaluar disfunción hepática ([Guerra-Ruiz et al., 2021](https://doi.org/10.1515/almed-2021-0016))
*  Se descarta `HCO3`, debido a su elevado porcentaje de valores ausentes y a su relación con otras variables del equilibrio ácido-base, como `BaseExcess` y `pH`. De forma similar, se descarta `SaO2` por su alta ausencia y su solapamiento parcial con `O2Sat`, que presenta mayor disponibilidad.

*   La variable `Hgb` se descarta por su elevada correlación con `Hct`, conservando esta última para representar información hematológica. Además, se descartan `AST`, `Alkalinephos`, `Fibrinogen` y `PTT` debido a su alto porcentaje de valores ausentes y a su menor prioridad dentro del modelo base.

El resto de variables de laboratorio se conservan por su posible relevancia clínica. Para estas variables se aplicará una estrategia combinada basada en imputación temporal mediante forward fill dentro de cada paciente e imputación final con estadísticos calculados únicamente sobre el conjunto de entrenamiento. Además, en aquellas variables con un porcentaje elevado de ausencia o con ausencia clínicamente informativa, se crearán indicadores binarios de ausencia.

## Signos vitales
Se conservan casi todos los signos vitales, pues la mayoría son variables continuas y son útiles para monitorización. Solo se descarta `EtCO2` pues presenta una ausencia muy alta. Para el resto de signos vitales se aplicará imputación temporal mediante forward fill y, cuando sea necesario, imputación final con la mediana calculada sobre el conjunto de entrenamiento.

## Variables demográficas y administrativas
Se descartan `Unit1` y `Unit2`, ya que son variables administrativas que identifican el tipo de unidad de cuidados intensivos y podrían introducir información dependiente de la organización hospitalaria más que del estado clínico del paciente.

Las variables `Age` y `Gender` se conservan sin imputación, al no presentar problemas relevantes de ausencia. La variable `HospAdmTime` se conserva y sus valores ausentes se imputarán mediante la mediana calculada en el conjunto de entrenamiento. Por último, la variable `ICULOS` se conserva porque representa el tiempo transcurrido desde el ingreso en UCI y aporta información temporal disponible en cada instante. No obstante, se plantea realizar experimentos con y sin esta variable para evaluar su impacto en el rendimiento del modelo.







### Eliminación de variables descartadas

In [ ]:
# Variables descartadas según lo descrito anteriormente
cols_to_drop_model = [
    "Hgb",
    "AST",
    "Alkalinephos",
    "Fibrinogen",
    "PTT",
    "EtCO2",
    "Unit1",
    "Unit2",
    "TroponinI",
    "Bilirubin_direct",
    "HCO3",
    "SaO2"
]

df_train = df_train.drop(columns=cols_to_drop_model)
df_test = df_test.drop(columns=cols_to_drop_model)

# Redefinimos las columnas que se utilizan como predictoras en el modelo
feature_cols = [col for col in feature_cols if col not in cols_to_drop_model]

# Revisar outliers en el conjunto de entrenamiento

In [ ]:
outliers_iqr = []

for var in feature_cols:
    serie = df_train[var].dropna()

    # Encontramos los cuartiles Q1 y Q3
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)

    #Calculamos el rango intercuartílico (IQR)
    iqr = q3 - q1

    # Establecemos los límites dentro de los cuales deberían encontrarse los
    # valores "típicos"
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    n_outliers = ((serie < limite_inferior) | (serie > limite_superior)).sum()
    porcentaje_outliers = n_outliers / len(serie) * 100

    outliers_iqr.append({
        "Variable": var,
        "Q1": q1,
        "Q3": q3,
        "Límite inferior IQR": limite_inferior,
        "Límite superior IQR": limite_superior,
        "Nº outliers": n_outliers,
        "% outliers": porcentaje_outliers
    })

outliers_iqr = pd.DataFrame(outliers_iqr)
outliers_iqr.sort_values("% outliers", ascending=False).round(2)


,Variable,Q1,Q3,Límite inferior IQR,Límite superior IQR,Nº outliers,% outliers
12,Calcium,7.70,8.80,6.05,10.45,10270,14.06
26,HospAdmTime,-46.84,-0.04,-117.04,70.16,170378,13.73
20,Bilirubin_total,0.50,1.70,-1.30,3.50,2288,12.32
14,Creatinine,0.70,1.45,-0.43,2.58,8960,11.85
8,FiO2,0.40,0.60,0.10,0.90,10425,10.05
16,Lactate,1.27,3.00,-1.32,5.60,2917,8.76
11,BUN,12.00,29.00,-13.50,54.50,6493,7.62
7,BaseExcess,-3.00,1.00,-9.00,7.00,3798,5.62
15,Glucose,106.00,153.00,35.50,223.50,11861,5.59
18,Phosphate,2.70,4.10,0.60,6.20,2595,5.21


In [ ]:
outlier_cols = ["Calcium", "Bilirubin_total", "Creatinine", "FiO2", "Lactate","BUN","BaseExcess", "Glucose", "Phosphate"]

df_train[outlier_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,Calcium,Bilirubin_total,Creatinine,FiO2,Lactate,BUN,BaseExcess,Glucose,Phosphate
count,73037.000000,18574.000000,75641.000000,103781.000000,33300.000000,85199.000000,67640.000000,212313.000000,49834.000000
mean,7.562918,2.143566,1.528030,0.562057,2.659402,23.954348,-0.679482,136.860804,3.551474
std,2.432142,4.339492,1.844584,12.416445,2.548440,20.042678,4.242816,50.881769,1.428110
min,1.000000,0.100000,0.100000,0.000000,0.200000,1.000000,-30.000000,13.000000,0.300000
1%,1.080000,0.200000,0.380000,0.210000,0.600000,4.000000,-13.000000,65.000000,1.200000
5%,1.170000,0.300000,0.500000,0.350000,0.800000,6.000000,-7.000000,82.000000,1.800000
50%,8.300000,0.900000,0.950000,0.500000,1.820000,17.000000,0.000000,127.000000,3.300000
95%,9.500000,8.200000,5.010000,1.000000,7.360500,65.000000,6.000000,229.000000,6.200000
99%,10.500000,25.327000,9.882000,1.000000,14.400100,102.000000,11.000000,324.440000,8.500000
max,27.900000,49.600000,46.600000,4000.000000,31.000000,268.000000,49.500000,988.000000,18.800000


No se eliminan automáticamente los valores extremos porque pueden representar situaciones clínicas reales. En su lugar, únicamente se plantea la corrección de valores fisiológicamente imposibles o extremadamente anómalos, manteniendo el resto de valores por su posible valor predictivo.

El escalado robusto se aplicará únicamente en aquellos modelos sensibles a la escala de las variables, como regresión logística o SVM. Para modelos basados en árboles, como Random Forest o XGBoost, no resulta estrictamente necesario, ya que estos algoritmos dividen las variables mediante umbrales y no dependen de la escala absoluta de las características.

## Corrección de valores fisiológicamente imposibles o extremadamente anómalos

In [ ]:
cap_rules = {
    "Calcium": {"lower": None, "upper": 20},
    "Creatinine": {"lower": None, "upper": 20},
    "BaseExcess": {"lower": -32, "upper": 45},
    "Glucose": {"lower": None, "upper": 800},
    "Phosphate": {"lower": None, "upper": 18}
}

for col, limits in cap_rules.items():
    if col in df_train.columns:
        df_train[col] = df_train[col].clip(
            lower=limits["lower"],
            upper=limits["upper"]
        )

    if col in df_test.columns:
        df_test[col] = df_test[col].clip(
            lower=limits["lower"],
            upper=limits["upper"]
        )

Para la variable `FiO2`, hacemos un tratamiento especial porque se han encontrado valores especialmente inusuales. La variable FiO2 representa la fracción inspirada de oxígeno administrada al paciente. En condiciones habituales se expresa como fracción entre 0.21 y 1.0, aunque en algunos registros pueden aparecer valores expresados como porcentaje. Por ello, los valores entre 1 y 100 se transforman dividiéndolos entre 100, mientras que los valores fuera del rango fisiológicamente plausible se consideran erróneos y se tratan como ausentes.

In [ ]:
for data in [df_train, df_test]:
    if "FiO2" in data.columns:
        # Si aparecen valores entre 1 y 100, podrían estar expresados como porcentaje
        mask_percent = (data["FiO2"] > 1) & (data["FiO2"] <= 100)
        data.loc[mask_percent, "FiO2"] = data.loc[mask_percent, "FiO2"] / 100

        # Valores fuera del rango fisiológicamente plausible se tratan como ausentes
        mask_invalid = (data["FiO2"] < 0.21) | (data["FiO2"] > 1)
        data.loc[mask_invalid, "FiO2"] = np.nan

In [ ]:
# Lista de variables modificadas
cols_to_check = ["Calcium", "Creatinine", "BaseExcess", "Glucose", "Phosphate", "FiO2"]

print("--- RESUMEN ESTADÍSTICO EN TRAIN (POST-TRATAMIENTO) ---")
# Filtramos solo las columnas de interés que existan en el DataFrame
existing_cols = [col for col in cols_to_check if col in df_train.columns]
display(df_train[existing_cols].describe().loc[["min", "max", "count"]])

print("\n--- RESUMEN ESTADÍSTICO EN TEST (POST-TRATAMIENTO) ---")
existing_cols_test = [col for col in cols_to_check if col in df_test.columns]
display(df_test[existing_cols_test].describe().loc[["min", "max", "count"]])

--- RESUMEN ESTADÍSTICO EN TRAIN (POST-TRATAMIENTO) ---


,Calcium,Creatinine,BaseExcess,Glucose,Phosphate,FiO2
min,1.0,0.1,-30.0,13.0,0.3,0.21
max,20.0,20.0,45.0,800.0,18.0,1.00
count,73037.0,75641.0,67640.0,212313.0,49834.0,103452.00



--- RESUMEN ESTADÍSTICO EN TEST (POST-TRATAMIENTO) ---


,Calcium,Creatinine,BaseExcess,Glucose,Phosphate,FiO2
min,1.0,0.1,-32.0,10.0,0.2,0.21
max,20.0,20.0,45.0,800.0,14.2,1.00
count,18294.0,18975.0,16505.0,53203.0,12467.0,25518.00


Redefinimos el informe de valores ausentes.

In [ ]:
missing_train_model = df_train[feature_cols].isna().mean().sort_values(ascending=False) * 100

missing_summary_model = pd.DataFrame({
    "Variable": missing_train_model.index,
    "PorcentajeAusentesTrain": missing_train_model.values
})

missing_summary_model.head(20)

,Variable,PorcentajeAusentesTrain
0,Bilirubin_total,98.502816
1,Lactate,97.315806
2,Phosphate,95.983060
3,Chloride,95.465970
4,BaseExcess,94.547782
5,PaCO2,94.412282
6,Calcium,94.112749
7,Platelets,94.043024
8,Creatinine,93.902850
9,Magnesium,93.681908


# Crear indicadores de ausencia para variables con un número elevado de valores ausentes

Para variables con elevado porcentaje de ausencia, se incorporan indicadores binarios que permiten al modelo identificar si una medición estaba disponible o no.

In [ ]:
high_missing_cols = missing_summary_model.loc[
    missing_summary_model["PorcentajeAusentesTrain"] > 70,
    "Variable"
].tolist()

high_missing_cols = [col for col in high_missing_cols if col in feature_cols]

for col in high_missing_cols:
    df_train[f"{col}_missing"] = df_train[col].isna().astype(int)
    df_test[f"{col}_missing"] = df_test[col].isna().astype(int)

missing_indicator_cols = [f"{col}_missing" for col in high_missing_cols]

feature_cols = feature_cols + missing_indicator_cols

In [ ]:
df_train.head()
print("El dataset de entrenamiento tiene", df_train.shape[0], "filas y", df_train.shape[1], "columnas.")

El dataset de entrenamiento tiene 1240596 filas y 49 columnas.


# Aplicar imputación temporal con forward fill

Ordenamos los registros por id del paciente y para cada paciente ordenamos sus registros en orden temporal. De esta manera, podemos aplicar imputación hacia delante dentro de cada paciente, utilizando únicamente valores anteriores del mismo paciente.

In [ ]:
df_train = df_train.sort_values(["PatientID", "TimeStep"]).copy()
df_test = df_test.sort_values(["PatientID", "TimeStep"]).copy()

df_train_ffill = df_train.copy()
df_test_ffill = df_test.copy()

df_train_ffill[feature_cols] = (
    df_train_ffill
    .groupby("PatientID")[feature_cols]
    .ffill()
)

df_test_ffill[feature_cols] = (
    df_test_ffill
    .groupby("PatientID")[feature_cols]
    .ffill()
)

Vemos el porcentaje de valores faltantes después de aplicar imputación hacia delante.

In [ ]:
missing_before = df_train[feature_cols].isna().mean() * 100
missing_after = df_train_ffill[feature_cols].isna().mean() * 100

missing_comparison = pd.DataFrame({
    "AntesForwardFill": missing_before,
    "DespuesForwardFill": missing_after
}).sort_values("AntesForwardFill", ascending=False)

missing_comparison.head(20)

,AntesForwardFill,DespuesForwardFill
Bilirubin_total,98.502816,69.893906
Lactate,97.315806,70.202467
Phosphate,95.983060,42.804346
Chloride,95.465970,53.571832
BaseExcess,94.547782,68.067929
PaCO2,94.412282,55.815028
Calcium,94.112749,28.462287
Platelets,94.043024,22.349258
Creatinine,93.902850,21.387946
Magnesium,93.681908,27.972523


# Imputar los ausentes restantes con medianas del conjunto de entrenamiento

Después de la imputación hacia adelante, los nulos restantes se rellenan con la mediana de la variable. Las medianas se calculan únicamente sobre el conjunto de entrenamiento y se aplican posteriormente al conjunto de prueba para evitar la fuga de información.

In [ ]:
medianas_train = df_train_ffill[feature_cols].median()

df_train_imputed = df_train_ffill.copy()
df_test_imputed = df_test_ffill.copy()

df_train_imputed[feature_cols] = df_train_imputed[feature_cols].fillna(medianas_train)
df_test_imputed[feature_cols] = df_test_imputed[feature_cols].fillna(medianas_train)

In [ ]:
print("Nulos train:", df_train_imputed[feature_cols].isna().sum().sum())
print("Nulos test:", df_test_imputed[feature_cols].isna().sum().sum())

Nulos train: 0
Nulos test: 0


# Crear características temporales sencillas
Vamos a aplicar ingeniería de características mediante ventanas deslizantes (*Rolling Features*). Seleccionamos las  constantes vitales principales: `HR`, `O2Sat`, `Temp`, `SBP`, `MAP` y `Resp`. Para cada una de ellas, se calculará una variable nueva cuyo valor será el promedio de las últimas 6 horas de dicha variable.

In [ ]:
temporal_cols = ["HR", "O2Sat", "Temp", "SBP", "MAP", "Resp"]

df_train_final = df_train_imputed.sort_values(["PatientID", "TimeStep"]).copy()
df_test_final = df_test_imputed.sort_values(["PatientID", "TimeStep"]).copy()

for col in temporal_cols:
    df_train_final[f"{col}_rolling_mean_6h"] = (
        df_train_final
        .groupby("PatientID")[col]
        .transform(lambda x: x.rolling(window=6, min_periods=1).mean())
    )

    df_test_final[f"{col}_rolling_mean_6h"] = (
        df_test_final
        .groupby("PatientID")[col]
        .transform(lambda x: x.rolling(window=6, min_periods=1).mean())
    )

Además, añadimos también una variable que calcula, para las mismas variables que considerábamos anteriormente, para cada instante t el valor que tiene la variable en ese momento menos el valor que tenía en el instante t-1 (si existe).

In [ ]:
for col in temporal_cols:
    df_train_final[f"{col}_diff_1h"] = (
        df_train_final
        .groupby("PatientID")[col]
        .diff()
        .fillna(0)
    )

    df_test_final[f"{col}_diff_1h"] = (
        df_test_final
        .groupby("PatientID")[col]
        .diff()
        .fillna(0)
    )

# Preparar matrices finales para el modelado

Realizamos unas últimas comprobaciones finales antes de guardar.

In [ ]:
print("Nulos en train:", df_train_final.isna().sum().sum())
print("Nulos en test:", df_test_final.isna().sum().sum())

print("Columnas train:", df_train_final.shape[1])
print("Columnas test:", df_test_final.shape[1])

print("Columnas diferentes entre train y test:")
print(set(df_train_final.columns) ^ set(df_test_final.columns))

print("Variables descartadas presentes en train:")
print(set(cols_to_drop_model) & set(df_train_final.columns))

Nulos en train: 0
Nulos en test: 0
Columnas train: 61
Columnas test: 61
Columnas diferentes entre train y test:
set()
Variables descartadas presentes en train:
set()


In [ ]:
cols_no_predictoras = ["PatientID", "Hospital", "TimeStep", "SepsisLabel"]

X_train = df_train_final.drop(columns=cols_no_predictoras)
y_train = df_train_final["SepsisLabel"]

X_test = df_test_final.drop(columns=cols_no_predictoras)
y_test = df_test_final["SepsisLabel"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(1240596, 57) (1240596,)
(311614, 57) (311614,)


# Escalado de variables numéricas según el algoritmo

El escalado no se aplica como transformación universal del dataset preprocesado, ya que no todos los algoritmos lo requieren. Para modelos sensibles a la escala, como regresión logística, SVM o KNN, se utilizará RobustScaler ajustado únicamente sobre el conjunto de entrenamiento. Para modelos basados en árboles, como Random Forest o XGBoost, no se aplicará escalado en la versión base. Por ello, guardamos de forma independiente el dataset escalado y el dataset no escalado.

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Columnas binarias de indicadores de ausencia
missing_indicator_cols = [col for col in X_train.columns if col.endswith("_missing")]

# Otras columnas binarias que no queremos escalar
binary_cols = missing_indicator_cols + [
    col for col in ["Gender"] if col in X_train.columns
]

# Columnas numéricas continuas a escalar
numeric_cols_to_scale = [
    col for col in X_train.columns
    if col not in binary_cols
]

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("robust_scaler", RobustScaler(), numeric_cols_to_scale),
        ("binary_features", "passthrough", binary_cols)
    ],
    remainder="drop"
)

In [ ]:
X_train_scaled = preprocessor_scaled.fit_transform(X_train)
X_test_scaled = preprocessor_scaled.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((1240596, 57), (311614, 57))

# Guardar el conjunto de datos preprocesado

In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

df_train_final.to_parquet(OUTPUT_DIR / "train_preprocessed.parquet", index=False)
df_test_final.to_parquet(OUTPUT_DIR / "test_preprocessed.parquet", index=False)

missing_summary.to_csv(OUTPUT_DIR / "missing_decisions.csv", index=False)

print("Datasets preprocesados sin escalar guardados correctamente.")

Datasets preprocesados sin escalar guardados correctamente.


In [ ]:
import joblib
# Reconstruir datasets escalados
scaled_feature_names = numeric_cols_to_scale + binary_cols

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=scaled_feature_names,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=scaled_feature_names,
    index=X_test.index
)

df_train_scaled = X_train_scaled_df.copy()
df_train_scaled["SepsisLabel"] = y_train.values

df_test_scaled = X_test_scaled_df.copy()
df_test_scaled["SepsisLabel"] = y_test.values

# Guardar datasets escalados por separado
df_train_scaled.to_parquet(
    OUTPUT_DIR / "train_preprocessed_scaled.parquet",
    index=False
)

df_test_scaled.to_parquet(
    OUTPUT_DIR / "test_preprocessed_scaled.parquet",
    index=False
)

# Guardar el transformador ajustado
joblib.dump(
    preprocessor_scaled,
    OUTPUT_DIR / "robust_scaler_preprocessor.joblib"
)

print("Datasets escalados guardados correctamente.")

Datasets escalados guardados correctamente.


# Referencias

* Guzmán, A. M., & Quiroga, T. (2010). Troponina en el diagnóstico de infarto al miocardio: Consideraciones desde el laboratorio clínico. *Revista médica de Chile*, *138*(3), 379-382. http://dx.doi.org/10.4067/S0034-98872010000300020
* Guerra-Ruiz, A. R., Crespo, J., López Martínez, R. M., Iruzubieta, P., Casals Mercadal, G., Lalana Garcés, M., Lavin Gomez, B. A., & Morales Ruiz, M. (2021). Bilirrubina: Medición y utilidad clínica en la enfermedad hepática. *Advances in Laboratory Medicine / Avances en Medicina de Laboratorio*, *2*(3), 362–372. https://doi.org/10.1515/almed-2021-0016
